### This file is merely used to test the vector store retrieval capability.

In [7]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

from langchain.vectorstores import Chroma
from chromadb.config import Settings
from chromadb import Client, PersistentClient
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Now using nomic-embed-text-v2-moe
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

query = "What is EIE4122?"
embedding = embedding_function.embed_query(query)

In [8]:
client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

collections = client.get_collection(collection_name)
docs = collections.get(
    include=["documents", "metadatas"],
    limit=collections.count()
)

docs_for_bm25 = [
    Document(page_content=doc_text, metadata=md)
    for doc_text, md in zip(docs["documents"], docs["metadatas"])
]

bm25_retriever = BM25Retriever.from_documents(docs_for_bm25)
print(f"BM25 retriever: initialized over {len(docs_for_bm25)} documents")

BM25 retriever: initialized over 15867 documents


In [9]:
docs = vectorStore.similarity_search(query, k=20)

for doc in docs:
    '''
    if "original_table" in doc.metadata:
        print("[Swapping for Raw Markdown Table]")
        new_result = doc.metadata["original_table"]
    else:
        new_result = doc.page_content
    '''
    
    print("========================================================")
    print(f"Content: {doc.page_content}...")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Chunk ID: {doc.metadata.get('chunk_id')}\n")

Content: --- Programme booklet: BEng/BSc (Hons) Scheme in Information and Artificial Intelligence Engineering | BEng/BSc (Hons) in IAIE: ['Subject Description Form'] ---

Subject Title, EIE4122 = Deep Learning and Deep Neural Networks. Credit Value, EIE4122 = 3. Level, EIE4122 = 4. Pre-requisite/ Co-requisite/ Exclusion, EIE4122 = EIE3124: Fundamentals of Machine Intelligence. Objectives, EIE4122 = This course is for students who would like to equip themselves with cutting-edge AI knowledge and know-howtojoin the AIprofession. Students will learn the foundations of deep learning and how to construct deep neural networks for real-world applications and AI systems. Students will also learn the trends in deep learning and deep neural networks.. Intended Subject Learning Outcomes, EIE4122 = Upon completion of the subject, students will be able to: Category A: Professional/academic knowledge and skills 1. Understand the benefits of deep learning and deep neural networks. 2. Understand the b

In [10]:
bm25_docs = bm25_retriever.get_relevant_documents(query)[:5]

for doc in bm25_docs:
    '''
    if "original_table" in doc.metadata:
        print("[Swapping for Raw Markdown Table]")
        new_result = doc.metadata["original_table"]
    else:
        new_result = doc.page_content
    '''
    
    print("========================================================")
    print(f"Content: {doc.page_content}...")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Chunk ID: {doc.metadata.get('chunk_id')}\n")

Content: --- PolyU Student Affairs Office (SAO) Website ---

Enquiry | Student Affairs Office

Email: stud.counselling@polyu.edu.hk
Phone: (852) 2766 6800
What is PolyU WellMind GO
Gift Redemption
Terms & Conditions
Read More
Survey...
Source: https://www.polyu.edu.hk/sao/counselling-and-wellness-section/programmes-and-activities/polyu-wellmind-go/enquiry/
Chunk ID: PolyU_SAO_enquiry_chunk_762

Content: --- PolyU Student Affairs Office (SAO) Website ---

• Gifts are available only on a first-come-first-served basis
• CWS reserves the right to make the final decision in the event of any dispute
• PolyU WellMind GO Terms and Conditions apply
What is PolyU WellMind GO
Terms & Conditions
Read More
Enquiry
Survey...
Source: https://www.polyu.edu.hk/sao/counselling-and-wellness-section/programmes-and-activities/polyu-wellmind-go/redemption/
Chunk ID: PolyU_SAO_redemption_chunk_520

Content: --- PolyU Student Affairs Office (SAO) Website ---

Is there any storage service in the Halls?
What is